# CRPSLoss & Probabilistic Forecasting

This notebook explains how to use `CRPSLoss` in BaseAttentive for
full-distribution probabilistic forecasting — going beyond point estimates
and simple quantile intervals.

## What is CRPS?

The **Continuous Ranked Probability Score (CRPS)** is a proper scoring
rule that measures how well a predicted probability distribution matches
the observed value.  Unlike:

- **MSE / MAE** — only score a single point prediction,
- **Pinball / quantile loss** — scores individual quantile levels separately,

CRPS rewards *calibrated* distributions that are both sharp (narrow) and
centred on the truth.  It collapses to MAE for a deterministic forecast.

$$
\text{CRPS}(F, y) = \int_{-\infty}^{\infty} \left(F(z) - \mathbf{1}[z \geq y]\right)^2 dz
$$

## Three Modes in BaseAttentive

| Mode | How it works | Best for |
|------|-------------|----------|
| `"quantile"` | Pinball loss averaged over user-specified quantile levels | Any output head; fast |
| `"gaussian"` | Closed-form CRPS for a single Gaussian predictive distribution | Unimodal, near-Gaussian targets |
| `"mixture"` | Monte-Carlo CRPS estimate from a Gaussian mixture | Multi-modal / heavy-tailed targets |

## Setup

In [ ]:
import os

import numpy as np

# Optional: pick a specific Keras backend before importing
# os.environ["KERAS_BACKEND"] = "torch"   # or "tensorflow" / "jax"
from base_attentive import BaseAttentive, __version__
from base_attentive.components.heads import (
    GaussianHead,
    MixtureDensityHead,
)
from base_attentive.components.losses import CRPSLoss

print(f"BaseAttentive {__version__} probabilistic imports OK")

## Synthetic Dataset

We create a small dataset with three distinct input streams (static,
dynamic, future) and a two-dimensional forecast target.

In [ ]:
BATCH = 64
LOOKBACK = 24  # historical time steps
HORIZON = 12  # forecast horizon
STATIC_DIM = 4
DYN_DIM = 8
FUT_DIM = 4
OUTPUT_DIM = 2

rng = np.random.default_rng(42)

x_static = rng.standard_normal((BATCH, STATIC_DIM)).astype(
    "float32"
)
x_dynamic = rng.standard_normal(
    (BATCH, LOOKBACK, DYN_DIM)
).astype("float32")
x_future = rng.standard_normal(
    (BATCH, HORIZON, FUT_DIM)
).astype("float32")

# Ground-truth targets: (batch, horizon, output_dim)
y_true = rng.standard_normal(
    (BATCH, HORIZON, OUTPUT_DIM)
).astype("float32")

print("x_static :", x_static.shape)
print("x_dynamic:", x_dynamic.shape)
print("x_future :", x_future.shape)
print("y_true   :", y_true.shape)

---

## Mode 1 — `"quantile"` (Pinball Loss)

The `quantile` mode computes the **pinball loss** (also called the
quantile loss or check function) for every quantile level you specify,
then averages across levels and all other dimensions.

### Build a quantile model

Pass `quantiles` to `BaseAttentive`; the output shape becomes
`(batch, horizon, n_quantiles, output_dim)`.

In [ ]:
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

model_q = BaseAttentive(
    static_input_dim=STATIC_DIM,
    dynamic_input_dim=DYN_DIM,
    future_input_dim=FUT_DIM,
    output_dim=OUTPUT_DIM,
    forecast_horizon=HORIZON,
    quantiles=QUANTILES,
    embed_dim=32,
    num_heads=4,
    dropout_rate=0.1,
    name="QuantileModel",
)

# Forward pass to determine output shape
preds_q = model_q([x_static, x_dynamic, x_future])
print("Quantile output shape:", preds_q.shape)
# Expected: (64, 12, 5, 2) — batch, horizon, quantiles, output_dim

In [ ]:
# CRPSLoss in quantile mode
crps_q = CRPSLoss(mode="quantile", quantiles=QUANTILES)

# Compile and do a quick training step
model_q.compile(optimizer="adam", loss=crps_q)

history_q = model_q.fit(
    x=[x_static, x_dynamic, x_future],
    y=y_true,
    epochs=3,
    batch_size=16,
    verbose=1,
)
print(
    "\nFinal training loss (quantile):",
    history_q.history["loss"][-1],
)

### Reading quantile predictions

The output layout is `(batch, horizon, n_quantiles, output_dim)`.  To
extract the median (index 2 for the 0.5 quantile above):

In [ ]:
import numpy as np

preds_np = np.array(preds_q)  # detach from any backend tensor

median_idx = QUANTILES.index(0.5)
median_pred = preds_np[:, :, median_idx, :]  # (64, 12, 2)

lower_idx = QUANTILES.index(0.1)
upper_idx = QUANTILES.index(0.9)
lower_bound = preds_np[:, :, lower_idx, :]  # (64, 12, 2)
upper_bound = preds_np[:, :, upper_idx, :]  # (64, 12, 2)

interval_width = upper_bound - lower_bound

print("Median prediction shape:", median_pred.shape)
print(
    "80 %% PI width (mean over batch & horizon):",
    interval_width.mean().round(4),
)

---

## Mode 2 — `gaussian` via `GaussianHead`

BaseAttentive v2.1.0 keeps the core model focused on **point** and
**quantile** forecasting. For Gaussian CRPS workflows, attach a
`GaussianHead` to any latent feature tensor produced by your own
backbone or preprocessing pipeline.

Here we use a synthetic latent feature tensor with shape
`(batch, horizon, feature_dim)`.


In [ ]:
FEATURE_DIM = 16
latent_features = rng.standard_normal(
    (BATCH, HORIZON, FEATURE_DIM)
).astype("float32")

gaussian_head = GaussianHead(output_dim=OUTPUT_DIM)
gaussian_raw = gaussian_head(latent_features)
y_pred_g = {
    "loc": gaussian_raw["mean"],
    "scale": gaussian_raw["scale"],
}

print("Gaussian loc   shape:", y_pred_g["loc"].shape)
print("Gaussian scale shape:", y_pred_g["scale"].shape)

In [ ]:
crps_g = CRPSLoss(mode="gaussian")
loss_g = crps_g(y_true, y_pred_g)

print("Gaussian CRPS:", float(np.array(loss_g)))

### Inspecting mean and standard deviation


In [ ]:
loc_pred = np.array(y_pred_g["loc"])
sigma_pred = np.array(y_pred_g["scale"])

print("Mean shape :", loc_pred.shape)
print("Sigma shape:", sigma_pred.shape)
print(
    "Mean sigma (first batch element, first step):",
    sigma_pred[0, 0],
)

---

## Mode 3 — `mixture` via `MixtureDensityHead`

For multi-modal predictive distributions, use `MixtureDensityHead` and
pass its outputs to `CRPSLoss(mode="mixture")`. The head predicts:

- component weights — shape `(B, H, K, D)`
- component means — shape `(B, H, K, D)`
- component scales — shape `(B, H, K, D)`

This is useful when the target distribution may have multiple plausible
future states.


In [ ]:
N_COMPONENTS = 3  # number of Gaussian mixture components

mixture_head = MixtureDensityHead(
    output_dim=OUTPUT_DIM, num_components=N_COMPONENTS
)
mixture_raw = mixture_head(latent_features)
y_pred_m = {
    "loc": mixture_raw["means"],
    "scale": mixture_raw["scales"],
    "weights": mixture_raw["weights"],
}

print("Mixture loc     shape:", y_pred_m["loc"].shape)
print("Mixture scale   shape:", y_pred_m["scale"].shape)
print("Mixture weights shape:", y_pred_m["weights"].shape)

In [ ]:
crps_m = CRPSLoss(mode="mixture", mc_samples=128)
loss_m = crps_m(y_true, y_pred_m)

print("Mixture CRPS (128 samples):", float(np.array(loss_m)))

---

## Comparing the Three Modes

| | `quantile` | `gaussian` | `mixture` |
|-|-----------|-----------|----------|
| **How it is used in v2.1.0** | Directly in `BaseAttentive` | Via `GaussianHead` | Via `MixtureDensityHead` |
| **Distribution shape** | Any (implicit) | Unimodal | Multi-modal |
| **Computation** | Pinball sum | Closed-form | Monte Carlo |
| **Output layout** | `(B, H, Q, O)` | dict with `loc`, `scale` | dict with `loc`, `scale`, `weights` |
| **Training speed** | Fast | Fast | Slower (MC samples) |
| **Calibration** | Quantile-level | Gaussian assumption | Flexible |


## Choosing `mc_samples`

Higher `mc_samples` gives a lower-variance CRPS estimate but increases
memory and compute.  Typical values:

| Situation | Recommended |
|-----------|-------------|
| Quick experiment | 32–64 |
| Production training | 128–256 |
| Final evaluation | 512–1024 |

In [ ]:
# Evaluate with higher mc_samples on the same mixture output
crps_eval = CRPSLoss(mode="mixture", mc_samples=512)
eval_loss = crps_eval(y_true, y_pred_m)
print(
    f"Evaluation CRPS (mixture, 512 samples): {float(np.array(eval_loss)):.6f}"
)

## Custom Training Loop with CRPSLoss

For maximum control (e.g., gradient clipping, custom schedulers) you
can call `CRPSLoss` directly inside a `GradientTape`:

In [ ]:
# Example shown for the Torch backend — adapt for TF/JAX as needed.
# import torch
# optimizer = torch.optim.AdamW(model_q.parameters(), lr=1e-3)
#
# for epoch in range(5):
#     optimizer.zero_grad()
#     preds = model_q([x_static_t, x_dynamic_t, x_future_t])
#     loss  = crps_q(y_true_t, preds)
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(model_q.parameters(), 1.0)
#     optimizer.step()
#     print(f"Epoch {epoch+1}  loss={loss.item():.4f}")
print(
    "See comment above for a Torch GradientTape-style training loop."
)

---

## Next Steps

- [07_v2_spec_registry.ipynb](07_v2_spec_registry.ipynb) — declarative `BaseAttentiveSpec` and custom component registration
- [05_kernel_robust_networks.ipynb](05_kernel_robust_networks.ipynb) — kernel-robust training strategies
- [Full documentation](https://base-attentive.readthedocs.io/)
- [API reference: CRPSLoss](https://base-attentive.readthedocs.io/en/latest/api_reference.html)